# Chapter 12 - SVM Classification

Support Vector Machines (SVMs) are among the most elegant algorithms in machine learning.
The core idea is deceptively simple: find the hyperplane that separates two classes with the
**widest possible margin**. This geometric intuition leads to a powerful classifier that
generalises well, handles high-dimensional data gracefully, and — through the kernel trick —
can learn complex non-linear boundaries.

**Topics covered:**
- The maximum-margin hyperplane
- Support vectors and why they matter
- Hard margin vs soft margin classification
- The C parameter (regularisation strength)
- SVC in scikit-learn
- Visualising the margin and support vectors
- The kernel trick: mapping to higher dimensions without computing them
- Common kernels: linear, polynomial, RBF
- The gamma parameter for the RBF kernel
- Decision boundaries with different kernels
- GridSearchCV for hyperparameter tuning
- When SVMs shine (and when they don't)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
%matplotlib inline

---
## 1. The Core Idea: Maximum-Margin Hyperplane

Imagine you have two classes of points in 2D. Many lines could separate them, but which
line is *best*?

An SVM answers: **the line (hyperplane in higher dimensions) that maximises the distance
between itself and the nearest data points on each side**. That distance is called the
**margin**.

Why maximise the margin? Intuitively, a wider margin means the classifier is more
"confident" in the gap between classes. Statistically, a wider margin corresponds to
better generalisation — the classifier is less sensitive to small perturbations in new
data points.

Mathematically, for a weight vector **w** and bias *b*, the decision boundary is:

$$\mathbf{w} \cdot \mathbf{x} + b = 0$$

The margin is $\frac{2}{\|\mathbf{w}\|}$, so maximising the margin is equivalent to
minimising $\|\mathbf{w}\|$ (or $\frac{1}{2}\|\mathbf{w}\|^2$ for mathematical
convenience), subject to every point being on the correct side.

In [ ]:
# Generate linearly separable data
X_blob, y_blob = make_blobs(n_samples=100, centers=2, cluster_std=1.2, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: many possible separating lines
ax = axes[0]
ax.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap='bwr', edgecolors='k', s=50)
# Draw a few arbitrary separating lines
x_range = np.linspace(X_blob[:, 0].min() - 1, X_blob[:, 0].max() + 1, 100)
for slope, intercept, color in [(-0.6, 2.5, 'grey'), (-0.8, 3.0, 'grey'), (-0.4, 1.5, 'grey')]:
    ax.plot(x_range, slope * x_range + intercept, color=color, linestyle='--', alpha=0.6)
ax.set_title('Many lines can separate the classes')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

# Right: the SVM picks the maximum-margin one
ax = axes[1]
svm_demo = SVC(kernel='linear', C=1e6)  # very large C -> hard margin behaviour
svm_demo.fit(X_blob, y_blob)

ax.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap='bwr', edgecolors='k', s=50)

# Decision boundary and margins
w = svm_demo.coef_[0]
b = svm_demo.intercept_[0]
x_dec = np.linspace(X_blob[:, 0].min() - 1, X_blob[:, 0].max() + 1, 200)
y_dec = -(w[0] * x_dec + b) / w[1]
margin = 1 / np.linalg.norm(w)
y_upper = -(w[0] * x_dec + b - 1) / w[1]
y_lower = -(w[0] * x_dec + b + 1) / w[1]

ax.plot(x_dec, y_dec, 'k-', linewidth=2, label='Decision boundary')
ax.plot(x_dec, y_upper, 'k--', linewidth=1, label='Margin')
ax.plot(x_dec, y_lower, 'k--', linewidth=1)
ax.fill_between(x_dec, y_lower, y_upper, alpha=0.1, color='green')

# Highlight support vectors
ax.scatter(svm_demo.support_vectors_[:, 0], svm_demo.support_vectors_[:, 1],
           s=200, facecolors='none', edgecolors='gold', linewidths=2, label='Support vectors')
ax.set_title('SVM: maximum-margin hyperplane')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

---
## 2. Support Vectors

The data points that sit exactly on the margin boundaries are called **support vectors**.
They are the most "difficult" points — the ones closest to the other class.

Key insight: **only the support vectors determine the decision boundary**. If you removed
every other training point and retrained, you would get the *exact same* hyperplane. This
makes SVMs memory-efficient at prediction time and also explains their robustness —
outliers far from the boundary have zero influence.

Scikit-learn exposes them via `model.support_vectors_`.

In [ ]:
print(f"Number of training samples: {len(X_blob)}")
print(f"Number of support vectors:  {len(svm_demo.support_vectors_)}")
print(f"Support vector indices:     {svm_demo.support_}")
print(f"Support vectors per class:  {svm_demo.n_support_}")
print(f"\nSupport vector coordinates:\n{svm_demo.support_vectors_}")

---
## 3. Hard Margin vs Soft Margin

### Hard margin
The formulation above assumes the data are **perfectly linearly separable** — no point is
on the wrong side. This is called *hard margin* classification. It has two problems:
1. It only works if the data really are linearly separable.
2. It is extremely sensitive to outliers — one stray point can completely change the
   boundary.

### Soft margin
In practice we allow some points to violate the margin (or even be on the wrong side).
Each violation incurs a penalty proportional to how far the point has strayed. The
optimisation becomes:

$$\min_{\mathbf{w}, b} \; \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i$$

where $\xi_i \geq 0$ are *slack variables* measuring each point's margin violation, and
**C** controls how much we care about violations vs margin width.

In [ ]:
# Create data with some overlap so we can see the effect
X_overlap, y_overlap = make_blobs(n_samples=120, centers=2, cluster_std=2.0, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, C_val, title in zip(axes, [1e6, 0.1],
                             ['Hard margin (very large C)', 'Soft margin (small C)']):
    clf = SVC(kernel='linear', C=C_val)
    clf.fit(X_overlap, y_overlap)

    ax.scatter(X_overlap[:, 0], X_overlap[:, 1], c=y_overlap, cmap='bwr',
               edgecolors='k', s=50)

    w = clf.coef_[0]
    b = clf.intercept_[0]
    x_line = np.linspace(X_overlap[:, 0].min() - 1, X_overlap[:, 0].max() + 1, 200)
    y_line = -(w[0] * x_line + b) / w[1]
    y_up = -(w[0] * x_line + b - 1) / w[1]
    y_dn = -(w[0] * x_line + b + 1) / w[1]

    ax.plot(x_line, y_line, 'k-', linewidth=2)
    ax.plot(x_line, y_up, 'k--', linewidth=1)
    ax.plot(x_line, y_dn, 'k--', linewidth=1)
    ax.fill_between(x_line, y_dn, y_up, alpha=0.08, color='green')
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='gold', linewidths=2)
    ax.set_title(f'{title}\nSupport vectors: {len(clf.support_vectors_)}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

---
## 4. The C Parameter: Regularisation

**C** is the single most important hyperparameter for a linear SVM:

| C value | Margin width | Misclassifications | Behaviour |
|---------|-------------|--------------------|-----------|
| **Small** (e.g. 0.01) | Wide | More allowed | High bias, low variance — underfitting risk |
| **Large** (e.g. 1000) | Narrow | Fewer allowed | Low bias, high variance — overfitting risk |

Think of C as the price you pay for each misclassification. A cheap price (small C) means
you tolerate errors in exchange for a comfortable margin. An expensive price (large C)
forces the model to classify almost every training point correctly, even if the margin
shrinks to a sliver.

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, C_val in zip(axes.ravel(), C_values):
    clf = SVC(kernel='linear', C=C_val)
    clf.fit(X_overlap, y_overlap)

    # Create mesh for decision boundary
    x_min, x_max = X_overlap[:, 0].min() - 1, X_overlap[:, 0].max() + 1
    y_min, y_max = X_overlap[:, 1].min() - 1, X_overlap[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-1, 0, 1], alpha=0.15, colors=['blue', 'red'])
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k',
               linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
    ax.scatter(X_overlap[:, 0], X_overlap[:, 1], c=y_overlap, cmap='bwr',
               edgecolors='k', s=40)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=150, facecolors='none', edgecolors='gold', linewidths=2)
    ax.set_title(f'C = {C_val}  |  SVs: {len(clf.support_vectors_)}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Effect of the C parameter on margin and support vectors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. SVC in Scikit-Learn: Basic Classification Example

Let us put things together with a clean, end-to-end example using `make_blobs`.

In [ ]:
# Generate a 3-class dataset
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [ ]:
# Build a pipeline: scale features then classify
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear', C=1.0))
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

---
## 6. Visualising the Margin and Support Vectors on 2D Data

A helper function that draws the decision boundary, margin bands, and highlights the
support vectors. We will reuse this throughout the notebook.

In [ ]:
def plot_svm_decision_boundary(clf, X, y, ax=None, title='', resolution=300):
    """Plot the decision boundary, margin, and support vectors for a fitted SVC."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))

    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])

    # For binary classification, decision_function returns 1D
    # For multi-class OvO it returns shape (n_samples, n_classes*(n_classes-1)/2)
    if Z.ndim == 1:
        Z = Z.reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.3)
        ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k',
                   linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
    else:
        # Multi-class: just show the predicted regions
        Z_pred = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z_pred, alpha=0.25, cmap='Set3')

    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='k', s=50, zorder=3)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=180, facecolors='none', edgecolors='gold', linewidths=2,
               label=f'Support vectors ({len(clf.support_vectors_)})', zorder=4)
    ax.set_title(title)
    ax.legend(loc='best')
    return ax

In [ ]:
# Visualise on the 2-class blob data
svm_vis = SVC(kernel='linear', C=1.0)
svm_vis.fit(X_blob, y_blob)

fig, ax = plt.subplots(figsize=(9, 6))
plot_svm_decision_boundary(svm_vis, X_blob, y_blob, ax=ax,
                           title='Linear SVM on make_blobs data')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

---
## 7. The Kernel Trick: Mapping to Higher Dimensions

What if the data are **not** linearly separable? We could manually engineer polynomial
features (e.g. add $x_1^2$, $x_1 x_2$, $x_2^2$, ...) so that a linear separator in the
new high-dimensional space corresponds to a nonlinear boundary in the original space.

The **kernel trick** achieves this without ever explicitly computing the high-dimensional
coordinates. The SVM optimisation only needs **dot products** between data points. A kernel
function $K(\mathbf{x}_i, \mathbf{x}_j)$ computes the dot product in the transformed
space directly:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i) \cdot \phi(\mathbf{x}_j)$$

where $\phi$ is the (implicit) mapping to higher dimensions.

This is computationally brilliant: you get the power of a high- (even infinite-) dimensional
feature space at the cost of computing a single kernel evaluation per pair of points.

### Visualising the Idea

Consider `make_circles`: two concentric rings. A linear boundary cannot separate them,
but if we add a third dimension $z = x_1^2 + x_2^2$, the inner ring lifts up and becomes
linearly separable.

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=42)

fig = plt.figure(figsize=(14, 5))

# Original 2D space
ax1 = fig.add_subplot(121)
ax1.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='bwr', edgecolors='k', s=40)
ax1.set_title('Original 2D space — not linearly separable')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_aspect('equal')

# Lifted to 3D
ax2 = fig.add_subplot(122, projection='3d')
z = X_circ[:, 0]**2 + X_circ[:, 1]**2
ax2.scatter(X_circ[:, 0], X_circ[:, 1], z, c=y_circ, cmap='bwr', edgecolors='k', s=40)
ax2.set_title('Lifted to 3D — now separable!')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_1^2 + x_2^2$')

plt.tight_layout()
plt.show()

---
## 8. Common Kernels

Scikit-learn's `SVC` supports several kernels:

| Kernel | Formula | Key parameters | Typical use case |
|--------|---------|----------------|------------------|
| **linear** | $K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i \cdot \mathbf{x}_j$ | C | Linearly separable data, text classification |
| **poly** | $(\gamma\,\mathbf{x}_i \cdot \mathbf{x}_j + r)^d$ | C, degree, gamma, coef0 | Image processing, moderate nonlinearity |
| **rbf** | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | C, gamma | General-purpose nonlinear; default |
| **sigmoid** | $\tanh(\gamma\,\mathbf{x}_i \cdot \mathbf{x}_j + r)$ | C, gamma, coef0 | Neural-network-like; rarely used |

The **RBF** (Radial Basis Function) kernel is the most versatile and is sklearn's default.
It maps data into an infinite-dimensional space, so it can learn very complex boundaries.

In [ ]:
# Compare kernels on make_circles
kernels = [
    ('linear', SVC(kernel='linear', C=1.0)),
    ('poly (d=2)', SVC(kernel='poly', degree=2, C=1.0)),
    ('poly (d=3)', SVC(kernel='poly', degree=3, C=1.0)),
    ('rbf', SVC(kernel='rbf', C=1.0)),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, clf) in zip(axes, kernels):
    clf.fit(X_circ, y_circ)
    plot_svm_decision_boundary(clf, X_circ, y_circ, ax=ax,
                               title=f'Kernel: {name}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Different kernels on make_circles data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. The Gamma Parameter (RBF Kernel)

For the RBF kernel, **gamma** ($\gamma$) controls how far the influence of a single
training example reaches:

- **Small gamma** (e.g. 0.01): each point influences a *large* region -> smooth,
  simple boundary (risk of underfitting).
- **Large gamma** (e.g. 100): each point influences only its immediate neighbourhood
  -> wiggly, complex boundary that wraps tightly around individual points (risk of
  overfitting).

Mathematically, large gamma makes the Gaussian bell curve narrow, so distant points
contribute almost nothing — the model effectively memorises the training set.

In [ ]:
X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)

gamma_values = [0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))

for ax, g in zip(axes, gamma_values):
    clf = SVC(kernel='rbf', gamma=g, C=1.0)
    clf.fit(X_moon, y_moon)
    plot_svm_decision_boundary(clf, X_moon, y_moon, ax=ax,
                               title=f'gamma = {g}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Effect of gamma on RBF kernel decision boundary (make_moons)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Notice how `gamma=0.01` produces almost a linear cut (underfitting), while `gamma=100`
creates tiny islands around each point (overfitting). The sweet spot is somewhere in
between.

---
## 10. C and Gamma Together

C and gamma interact. Both influence model complexity, but from different angles:

- **C** controls tolerance for misclassification.
- **gamma** controls the shape of the decision boundary.

Together they form a 2D landscape of model complexity. Let us visualise a grid.

In [ ]:
C_vals = [0.01, 1, 100]
gamma_vals = [0.1, 1, 10]

fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for i, g in enumerate(gamma_vals):
    for j, c in enumerate(C_vals):
        ax = axes[i, j]
        clf = SVC(kernel='rbf', C=c, gamma=g)
        clf.fit(X_moon, y_moon)
        plot_svm_decision_boundary(clf, X_moon, y_moon, ax=ax,
                                   title=f'C={c}, gamma={g}')
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')

# Label the rows and columns
for ax, g in zip(axes[:, 0], gamma_vals):
    ax.set_ylabel(f'gamma={g}\n$x_2$', fontsize=11)

plt.suptitle('C vs gamma: joint effect on the RBF SVM decision boundary',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. Decision Boundaries on make_moons and make_circles

A side-by-side comparison of how different kernels handle two classic nonlinear datasets.

In [ ]:
datasets = [
    ('make_moons', make_moons(n_samples=200, noise=0.15, random_state=42)),
    ('make_circles', make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=42)),
]

kernel_configs = [
    ('linear', dict(kernel='linear', C=1)),
    ('poly d=3', dict(kernel='poly', degree=3, C=1)),
    ('rbf', dict(kernel='rbf', gamma='scale', C=1)),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (ds_name, (X_ds, y_ds)) in enumerate(datasets):
    for col, (k_name, params) in enumerate(kernel_configs):
        ax = axes[row, col]
        clf = SVC(**params)
        clf.fit(X_ds, y_ds)
        plot_svm_decision_boundary(clf, X_ds, y_ds, ax=ax,
                                   title=f'{ds_name} — {k_name}')
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

The RBF kernel handles both datasets beautifully. The linear kernel fails on both
because the data are inherently nonlinear. The polynomial kernel with degree 3 does
well on moons but struggles somewhat with circles.

---
## 12. GridSearchCV for C and Gamma Tuning

In practice we do not pick C and gamma by eye — we use cross-validated grid search.
A `Pipeline` with `StandardScaler` ensures feature scaling is done correctly inside
each fold (no data leakage).

In [ ]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moon, y_moon, test_size=0.25, random_state=42
)

pipe_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

param_grid = {
    'svc__C': [0.01, 0.1, 1, 10, 100],
    'svc__gamma': [0.01, 0.1, 1, 10],
    'svc__kernel': ['rbf'],
}

grid = GridSearchCV(pipe_grid, param_grid, cv=5, scoring='accuracy',
                    return_train_score=True, n_jobs=-1)
grid.fit(X_train_m, y_train_m)

print(f"Best parameters:  {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Test accuracy:    {grid.score(X_test_m, y_test_m):.4f}")

In [ ]:
# Visualise the grid search results as a heatmap
import pandas as pd

results = pd.DataFrame(grid.cv_results_)
pivot = results.pivot_table(
    index='param_svc__gamma',
    columns='param_svc__C',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('C')
ax.set_ylabel('gamma')
ax.set_title('GridSearchCV Mean Test Accuracy')

# Annotate cells
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.3f}',
                ha='center', va='center', fontsize=10,
                color='white' if pivot.values[i, j] < 0.85 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the best model's decision boundary
best_svc = grid.best_estimator_.named_steps['svc']
X_test_scaled = grid.best_estimator_.named_steps['scaler'].transform(X_test_m)
X_all_scaled = grid.best_estimator_.named_steps['scaler'].transform(X_moon)

fig, ax = plt.subplots(figsize=(9, 6))
plot_svm_decision_boundary(best_svc, X_all_scaled, y_moon, ax=ax,
                           title=f'Best model: C={grid.best_params_["svc__C"]}, '
                                 f'gamma={grid.best_params_["svc__gamma"]}')
ax.set_xlabel('$x_1$ (scaled)')
ax.set_ylabel('$x_2$ (scaled)')
plt.tight_layout()
plt.show()

---
## 13. Multi-Class Classification with SVM

SVMs are inherently binary classifiers, but scikit-learn handles multi-class problems
automatically using the **one-vs-one (OvO)** strategy by default. For *k* classes, this
trains $\frac{k(k-1)}{2}$ binary classifiers and each test point is classified by a
majority vote.

In [ ]:
# 4-class blob dataset
X_multi, y_multi = make_blobs(n_samples=400, centers=4, cluster_std=1.5, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X_multi, y_multi, test_size=0.25, random_state=42)

clf_multi = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', C=10, gamma='scale'))
])
clf_multi.fit(X_tr, y_tr)

print(f"Test accuracy: {clf_multi.score(X_te, y_te):.4f}")
print()
print(classification_report(y_te, clf_multi.predict(X_te)))

In [ ]:
# Visualise the multi-class decision regions
svc_inner = clf_multi.named_steps['svc']
scaler = clf_multi.named_steps['scaler']
X_multi_sc = scaler.transform(X_multi)

fig, ax = plt.subplots(figsize=(9, 7))

x_min, x_max = X_multi_sc[:, 0].min() - 1, X_multi_sc[:, 0].max() + 1
y_min, y_max = X_multi_sc[:, 1].min() - 1, X_multi_sc[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = svc_inner.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.25, cmap='Set2')
scatter = ax.scatter(X_multi_sc[:, 0], X_multi_sc[:, 1], c=y_multi,
                     cmap='Set2', edgecolors='k', s=40)
ax.scatter(svc_inner.support_vectors_[:, 0], svc_inner.support_vectors_[:, 1],
           s=150, facecolors='none', edgecolors='gold', linewidths=2,
           label=f'Support vectors ({len(svc_inner.support_vectors_)})')
ax.set_title('Multi-class SVM (OvO) decision regions')
ax.set_xlabel('$x_1$ (scaled)')
ax.set_ylabel('$x_2$ (scaled)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

---
## 14. Feature Scaling Matters

SVMs measure distances between data points (especially the RBF kernel). If one feature
ranges from 0 to 1 and another from 0 to 10,000, the second feature will dominate.
**Always scale your features** before training an SVM — `StandardScaler` or `MinMaxScaler`
are the standard choices.

Use a `Pipeline` so that scaling is applied consistently in cross-validation and on new
data.

In [ ]:
# Demonstrate the impact of scaling
X_scale_demo = X_moon.copy()
X_scale_demo[:, 1] = X_scale_demo[:, 1] * 1000  # blow up second feature

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without scaling
clf_noscale = SVC(kernel='rbf', C=1, gamma='scale')
clf_noscale.fit(X_scale_demo, y_moon)
axes[0].scatter(X_scale_demo[:, 0], X_scale_demo[:, 1], c=y_moon, cmap='bwr',
                edgecolors='k', s=40)
axes[0].set_title(f'Without explicit scaling\nAccuracy: {clf_noscale.score(X_scale_demo, y_moon):.3f}')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$ (x1000)')

# With scaling
scaler_demo = StandardScaler()
X_scale_std = scaler_demo.fit_transform(X_scale_demo)
clf_scaled = SVC(kernel='rbf', C=1, gamma='scale')
clf_scaled.fit(X_scale_std, y_moon)
axes[1].scatter(X_scale_std[:, 0], X_scale_std[:, 1], c=y_moon, cmap='bwr',
                edgecolors='k', s=40)
axes[1].set_title(f'With StandardScaler\nAccuracy: {clf_scaled.score(X_scale_std, y_moon):.3f}')
axes[1].set_xlabel('$x_1$ (scaled)')
axes[1].set_ylabel('$x_2$ (scaled)')

plt.suptitle('Feature scaling is critical for SVM', fontsize=14)
plt.tight_layout()
plt.show()

---
## 15. When SVMs Shine and Their Limitations

### Strengths

- **Effective in high-dimensional spaces** — SVMs work well even when the number of
  features exceeds the number of samples (e.g. text classification, genomics).
- **Memory-efficient** — only the support vectors are stored, not the entire training set.
- **Versatile** — the kernel trick lets you model complex nonlinear boundaries without
  explicit feature engineering.
- **Good generalisation** — the maximum-margin principle acts as built-in regularisation.

### Limitations

- **Slow on large datasets** — training complexity is roughly $O(n^2)$ to $O(n^3)$, so
  SVMs struggle with hundreds of thousands of samples. For large datasets consider
  `LinearSVC` (which uses a different optimisation algorithm) or SGDClassifier with
  `loss='hinge'`.
- **Sensitive to feature scaling** — as demonstrated above.
- **No direct probability estimates** — `SVC` can produce probabilities via Platt scaling
  (`probability=True`), but this is expensive and not always well-calibrated.
- **Not great for very noisy data** — if classes overlap heavily, simpler models
  (logistic regression, random forests) may be more practical.
- **Hyperparameter tuning required** — C, gamma, and kernel choice all matter and
  interact with each other.

---
## 16. Practical Workflow Summary

A recommended workflow for SVM classification:

1. **Scale features** — use `StandardScaler` inside a `Pipeline`.
2. **Start with RBF kernel** — it is the most flexible default.
3. **Grid search over C and gamma** — use `GridSearchCV` with logarithmic ranges
   (e.g. `[0.01, 0.1, 1, 10, 100]`).
4. **Evaluate on a held-out test set** — never tune on test data.
5. **Consider a linear kernel** if the dataset is very large or high-dimensional —
   it is faster and often sufficient.
6. **Inspect support vectors** — if almost all points are support vectors, the model
   may be underfitting (C too small or wrong kernel).

In [ ]:
# Complete end-to-end example: make_moons with full pipeline
X_final, y_final = make_moons(n_samples=500, noise=0.2, random_state=0)
X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(X_final, y_final,
                                                    test_size=0.2, random_state=42)

# Pipeline
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

# Param grid
param_grid_full = {
    'svc__kernel': ['linear', 'rbf', 'poly'],
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.1, 1, 10],
}

gs = GridSearchCV(svm_pipe, param_grid_full, cv=5, scoring='accuracy',
                  n_jobs=-1, return_train_score=True)
gs.fit(X_tr_f, y_tr_f)

print(f"Best parameters:  {gs.best_params_}")
print(f"Best CV score:    {gs.best_score_:.4f}")
print(f"Test accuracy:    {gs.score(X_te_f, y_te_f):.4f}")

In [ ]:
# Final visualisation
best_model = gs.best_estimator_
X_final_sc = best_model.named_steps['scaler'].transform(X_final)
best_svc_final = best_model.named_steps['svc']

fig, ax = plt.subplots(figsize=(10, 7))
plot_svm_decision_boundary(best_svc_final, X_final_sc, y_final, ax=ax,
                           title=f'Best SVM on make_moons (n=500)\n'
                                 f'kernel={gs.best_params_["svc__kernel"]}, '
                                 f'C={gs.best_params_["svc__C"]}, '
                                 f'gamma={gs.best_params_["svc__gamma"]}')
ax.set_xlabel('$x_1$ (scaled)')
ax.set_ylabel('$x_2$ (scaled)')
plt.tight_layout()
plt.show()

---

**Next up:** [02 - SVM Regression](02_svm_regression.ipynb) — applying the same
margin-based intuition to regression problems with SVR.